# Video Game Sales Prediction — Machine Learning Analysis
**Data Analyst Portfolio Project**  
Hugo Apolinário · 2025

**Business questions:**
1. Can we predict how many copies a game will sell globally? *(Regression)*
2. Can we predict whether a game will be a hit or a flop? *(Classification)*

This notebook builds and evaluates two machine learning models on 16,000+ video games,
translating model outputs into actionable insights for game publishers.

---
## 0. Setup — install and import libraries

In [ ]:
import micropip
await micropip.install('seaborn')
await micropip.install('scikit-learn')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# scikit-learn — the standard Python machine learning library
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_absolute_error, r2_score,
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

print('Libraries loaded successfully!')

---
## 1. Load and clean the data

In [ ]:
# Load the dataset
df = pd.read_csv('vgsales.csv')

# Clean — same steps as the SQL project
df.dropna(subset=['Year', 'Publisher'], inplace=True)
df['Year'] = df['Year'].astype(int)
df = df[df['Year'] <= 2016]
df.reset_index(drop=True, inplace=True)

print(f'Dataset loaded and cleaned: {len(df):,} rows')
print(f'Columns: {list(df.columns)}')
df.head()

---
## 2. Feature engineering
Machine learning models need numbers — not text. We convert categorical columns
(Platform, Genre, Publisher) into numeric codes using Label Encoding.

In [ ]:
# Make a copy for ML — we don't want to change the original
ml_df = df.copy()

# Label encode categorical columns
# This converts e.g. 'Action' -> 0, 'Sports' -> 1, 'Shooter' -> 2 etc.
le = LabelEncoder()
ml_df['Platform_encoded'] = le.fit_transform(ml_df['Platform'])
ml_df['Genre_encoded']    = le.fit_transform(ml_df['Genre'])
ml_df['Publisher_encoded'] = le.fit_transform(ml_df['Publisher'])

# Define our features (X) and target (y)
# Features = what we know about a game before it launches
# Target = what we want to predict (Global Sales)
FEATURES = ['Platform_encoded', 'Genre_encoded', 'Publisher_encoded',
            'Year', 'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']

X = ml_df[FEATURES]
y = ml_df['Global_Sales']

print('Features used for prediction:')
for f in FEATURES:
    print(f'  - {f}')
print(f'\nTarget: Global_Sales')
print(f'Total samples: {len(X):,}')

---
## PART 1 — Regression: Predicting global sales
### 3. Train/test split

In [ ]:
# Split data into training set (80%) and test set (20%)
# The model learns from training data and is evaluated on test data
# random_state=42 ensures reproducible results
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {len(X_train):,} games')
print(f'Test set:     {len(X_test):,} games')
print(f'\nThe model will learn from {len(X_train):,} games')
print(f'and be tested on {len(X_test):,} games it has never seen.')

### 4. Model 1 — Linear Regression

In [ ]:
# Linear Regression — simple, fast, interpretable
# Finds the best straight line through the data
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)

print('LINEAR REGRESSION RESULTS')
print(f'  MAE (Mean Absolute Error): {mae_lr:.3f} million units')
print(f'  R² Score:                  {r2_lr:.3f}')
print(f'\n  Interpretation:')
print(f'  On average the model is off by {mae_lr:.2f} million sales')
print(f'  R² of {r2_lr:.2f} means the model explains {r2_lr*100:.1f}% of sales variance')

### 5. Model 2 — Random Forest Regressor

In [ ]:
# Random Forest — more powerful, handles complex patterns
# Builds 100 decision trees and averages their predictions
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf  = r2_score(y_test, y_pred_rf)

print('RANDOM FOREST REGRESSOR RESULTS')
print(f'  MAE (Mean Absolute Error): {mae_rf:.3f} million units')
print(f'  R² Score:                  {r2_rf:.3f}')
print(f'\n  Interpretation:')
print(f'  On average the model is off by {mae_rf:.2f} million sales')
print(f'  R² of {r2_rf:.2f} means the model explains {r2_rf*100:.1f}% of sales variance')

# Compare models
print('\nMODEL COMPARISON')
print(f'  Linear Regression MAE : {mae_lr:.3f}')
print(f'  Random Forest MAE     : {mae_rf:.3f}')
winner = 'Random Forest' if mae_rf < mae_lr else 'Linear Regression'
print(f'  Winner: {winner}')

### 6. Feature importance — what predicts sales most?

In [ ]:
# Feature importance tells us which variables the model relies on most
# This is one of the most valuable outputs for a business audience
importance_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': rf_reg.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    importance_df['Feature'],
    importance_df['Importance'],
    color=sns.color_palette('Blues_r', len(importance_df))
)
for bar in bars:
    ax.text(bar.get_width() + 0.002,
            bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}',
            va='center', fontsize=9)

ax.set_title('Feature importance — what predicts global sales?',
             fontsize=14, pad=15)
ax.set_xlabel('Importance score')
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
plt.tight_layout()
plt.savefig('chart_01_feature_importance.png')
plt.show()
print('Chart saved!')

### 7. Predicted vs actual sales

In [ ]:
# Plot predicted vs actual — a perfect model would show all points on the diagonal
fig, ax = plt.subplots(figsize=(9, 7))

ax.scatter(y_test, y_pred_rf, alpha=0.4, color='#2E86AB',
           edgecolors='white', linewidth=0.3, s=30)

# Perfect prediction line
max_val = max(y_test.max(), y_pred_rf.max())
ax.plot([0, max_val], [0, max_val], 'r--',
        linewidth=1.5, label='Perfect prediction')

ax.set_title('Random Forest — predicted vs actual global sales',
             fontsize=14, pad=15)
ax.set_xlabel('Actual global sales (millions)')
ax.set_ylabel('Predicted global sales (millions)')
ax.legend()
plt.tight_layout()
plt.savefig('chart_02_predicted_vs_actual.png')
plt.show()
print('Chart saved!')

---
## PART 2 — Classification: Hit or flop?
### 8. Create the target label

In [ ]:
# Define a HIT as a game in the top 25% of global sales
# Everything else is a FLOP
threshold = ml_df['Global_Sales'].quantile(0.75)
ml_df['Hit'] = (ml_df['Global_Sales'] >= threshold).astype(int)

hits  = ml_df['Hit'].sum()
flops = len(ml_df) - hits

print(f'Sales threshold for a HIT: {threshold:.2f} million copies')
print(f'Hits (top 25%): {hits:,} games')
print(f'Flops (bottom 75%): {flops:,} games')

# New features and target for classification
# Note: we exclude NA/EU/JP/Other sales to avoid data leakage
# (we wouldnt know regional sales before a game launches)
CLASS_FEATURES = ['Platform_encoded', 'Genre_encoded',
                  'Publisher_encoded', 'Year']

X_cls = ml_df[CLASS_FEATURES]
y_cls = ml_df['Hit']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)

print(f'\nClassification training set: {len(X_train_c):,} games')
print(f'Classification test set:     {len(X_test_c):,} games')

### 9. Model 3 — Logistic Regression

In [ ]:
# Logistic Regression — classic classification algorithm
# Despite the name it predicts categories, not numbers
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_c, y_train_c)
y_pred_log = log_reg.predict(X_test_c)

acc_log = accuracy_score(y_test_c, y_pred_log)

print('LOGISTIC REGRESSION RESULTS')
print(f'  Accuracy: {acc_log:.3f} ({acc_log*100:.1f}%)')
print(f'\nDetailed report:')
print(classification_report(y_test_c, y_pred_log,
                            target_names=['Flop', 'Hit']))

### 10. Model 4 — Random Forest Classifier

In [ ]:
# Random Forest Classifier — more powerful than Logistic Regression
rf_cls = RandomForestClassifier(n_estimators=100, random_state=42)
rf_cls.fit(X_train_c, y_train_c)
y_pred_cls = rf_cls.predict(X_test_c)

acc_cls = accuracy_score(y_test_c, y_pred_cls)

print('RANDOM FOREST CLASSIFIER RESULTS')
print(f'  Accuracy: {acc_cls:.3f} ({acc_cls*100:.1f}%)')
print(f'\nDetailed report:')
print(classification_report(y_test_c, y_pred_cls,
                            target_names=['Flop', 'Hit']))

print('MODEL COMPARISON')
print(f'  Logistic Regression accuracy : {acc_log*100:.1f}%')
print(f'  Random Forest accuracy       : {acc_cls*100:.1f}%')
winner_cls = 'Random Forest' if acc_cls > acc_log else 'Logistic Regression'
print(f'  Winner: {winner_cls}')

### 11. Confusion matrix

In [ ]:
# Confusion matrix shows where the model gets it right and wrong
# True Positives: correctly predicted hits
# True Negatives: correctly predicted flops
# False Positives: predicted hit but was actually a flop
# False Negatives: predicted flop but was actually a hit

cm = confusion_matrix(y_test_c, y_pred_cls)

fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Flop', 'Hit']
)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion matrix — Random Forest Classifier',
             fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('chart_03_confusion_matrix.png')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correctly predicted flops): {tn:,}')
print(f'False Positives (predicted hit, was flop)  : {fp:,}')
print(f'False Negatives (predicted flop, was hit)  : {fn:,}')
print(f'True Positives  (correctly predicted hits) : {tp:,}')
print('Chart saved!')

### 12. Classification feature importance

In [ ]:
importance_cls = pd.DataFrame({
    'Feature': CLASS_FEATURES,
    'Importance': rf_cls.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    importance_cls['Feature'],
    importance_cls['Importance'],
    color=sns.color_palette('Reds_r', len(importance_cls))
)
for bar in bars:
    ax.text(bar.get_width() + 0.002,
            bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}',
            va='center', fontsize=9)

ax.set_title('Feature importance — what predicts a hit vs flop?',
             fontsize=14, pad=15)
ax.set_xlabel('Importance score')
ax.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
plt.tight_layout()
plt.savefig('chart_04_classification_importance.png')
plt.show()
print('Chart saved!')

---
## 13. Business recommendations

In [ ]:
top_feature_reg = importance_df.iloc[-1]['Feature']
top_feature_cls = importance_cls.iloc[-1]['Feature']

print('=' * 60)
print('  VIDEO GAME ML — BUSINESS RECOMMENDATIONS')
print('=' * 60)
print(f'''
1. REGIONAL SALES ARE THE STRONGEST PREDICTOR
   The regression model confirms that NA, EU and JP sales
   are the most predictive features. Publishers should
   monitor early regional sales data as the clearest
   signal of global performance.

2. PUBLISHER MATTERS MORE THAN PLATFORM FOR HITS
   The classification model shows Publisher is the strongest
   predictor of whether a game becomes a hit. Established
   publishers have a significant structural advantage —
   brand recognition, distribution, and marketing budgets
   all contribute to hit probability.

3. RANDOM FOREST OUTPERFORMS LINEAR MODELS
   In both tasks, Random Forest significantly outperformed
   the simpler linear model. This tells us that the
   relationship between game features and sales is non-linear
   and complex — simple rules of thumb are unreliable.

4. CLASSIFICATION IS MORE ACTIONABLE THAN REGRESSION
   Predicting exact sales is hard. Predicting hit vs flop
   is more reliable and more useful for go/no-go decisions.
   Publishers should use classification models for green-
   lighting decisions, not regression models.

5. GENRE AND PLATFORM ARE CONTROLLABLE LEVERS
   Unlike publisher reputation, genre and platform are
   decisions publishers can make. The feature importance
   analysis shows these still matter — choosing the right
   platform and genre combination improves hit probability.
''')
print('=' * 60)

---
## 14. Summary

In [ ]:
print('PROJECT SUMMARY')
print(f'  Dataset               : {len(ml_df):,} video games')
print(f'  Features used         : {len(FEATURES)} (regression), {len(CLASS_FEATURES)} (classification)')
print(f'  Train/test split      : 80% / 20%')
print()
print('REGRESSION RESULTS (predicting global sales)')
print(f'  Linear Regression MAE : {mae_lr:.3f}M | R²: {r2_lr:.3f}')
print(f'  Random Forest MAE     : {mae_rf:.3f}M | R²: {r2_rf:.3f}')
print(f'  Best model            : Random Forest')
print()
print('CLASSIFICATION RESULTS (predicting hit vs flop)')
print(f'  Hit threshold         : {threshold:.2f}M copies')
print(f'  Logistic Regression   : {acc_log*100:.1f}% accuracy')
print(f'  Random Forest         : {acc_cls*100:.1f}% accuracy')
print(f'  Best model            : Random Forest')